# CineMind-AI MovieLens 25M NCF candidate training

This notebook is an isolated candidate training handoff, not an automatic training job. It benchmarks real candidate-shard batches first. Full training remains disabled by default until you set the explicit opt-in below and enter the confirmation phrase. The output is a separately named candidate (`collaborative_ncf_candidate_20261011`); production artifacts are never promoted or overwritten.

### Getting Started
1. Export the candidate bundle locally using `training/colab_ncf/README.md` (or use the pre-verified archive `cinemind_ncf_candidate_20261011.zip`).
2. Select a Colab GPU runtime before executing (**Runtime → Change runtime type → T4 GPU**).
3. By default, this notebook runs in **Drive-free mode** (`USE_DRIVE = False`). Upload `cinemind_ncf_candidate_20261011.zip` directly to `/content/` via the Files panel (left sidebar). This avoids Google Drive permissions and credential propagation issues.
4. Note: `/content` is ephemeral and is completely erased when Colab disconnects or recycles the runtime. Follow the checkpoint download instructions below to preserve training progress.

In [ ]:
import importlib.util, subprocess, sys
required = {'numpy': 'numpy', 'pandas': 'pandas', 'joblib': 'joblib'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])

# Drive mounting is disabled by default (USE_DRIVE = False) to avoid credential propagation errors.
# Set USE_DRIVE = True only if you explicitly prefer using Google Drive storage.
USE_DRIVE = False
drive_mounted = False

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_mounted = True
        print('✓ Google Drive mounted successfully.')
    except Exception as exc:
        print(f'⚠️ Google Drive mount failed or was skipped: {exc}')
        print('Falling back to standalone Colab storage mode (/content/cinemind_run).')
else:
    print('Operating in standalone Drive-free mode (USE_DRIVE = False).')
    print('Upload cinemind_ncf_candidate_20261011.zip directly to /content/ via the Files sidebar.')

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU unavailable. Select Runtime > Change runtime type > GPU, then rerun.')
device = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)
print('Properties:', torch.cuda.get_device_properties(0))
print('GPU memory GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
from pathlib import Path
import os, sys

# Set bundle and run paths based on storage mode
if drive_mounted and Path('/content/drive/MyDrive').is_dir():
    DEFAULT_ZIP = Path('/content/drive/MyDrive/CineMind-NCF/cinemind_ncf_candidate_20261011.zip')
    DEFAULT_RUN = Path('/content/drive/MyDrive/CineMind-NCF/run_20261011')
else:
    DEFAULT_ZIP = Path('/content/cinemind_ncf_candidate_20261011.zip')
    DEFAULT_RUN = Path('/content/cinemind_run')

# If DEFAULT_ZIP is not in Drive, check /content fallback
if not DEFAULT_ZIP.is_file() and Path('/content/cinemind_ncf_candidate_20261011.zip').is_file():
    DEFAULT_ZIP = Path('/content/cinemind_ncf_candidate_20261011.zip')

BUNDLE_ZIP = DEFAULT_ZIP
RUN_DIR = DEFAULT_RUN
BUNDLE_DIR = Path('/content/cinemind_ncf_bundle')

if not BUNDLE_ZIP.is_file():
    raise FileNotFoundError(
        f'Candidate bundle ZIP not found at: {BUNDLE_ZIP}\n'
        'Please upload cinemind_ncf_candidate_20261011.zip to /content/ via Colab files panel (left sidebar),\n'
        'or place it in your Google Drive folder if USE_DRIVE is enabled, then rerun this cell.'
    )

import hashlib, json, zipfile
with zipfile.ZipFile(BUNDLE_ZIP) as zf:
    names = zf.namelist()
    if len(names) != len(set(names)) or 'manifest.json' not in names:
        raise ValueError('Invalid bundle member list or missing manifest')
    manifest = json.loads(zf.read('manifest.json'))
    if set(names) != set(manifest['files']) | {'manifest.json'}:
        raise ValueError('Bundle members do not match manifest allow-list')
    candidate_names = {name for name in names if name.startswith('candidate/')}
    source_names = {name for name in names if name.startswith('source/')}
    expected_source_names = {
        'source/app/__init__.py', 'source/app/ml/__init__.py',
        'source/app/ml/model_store.py', 'source/app/ml/ncf_data.py',
        'source/app/ml/ncf_evaluation.py', 'source/app/ml/ncf_model.py',
        'source/app/ml/ncf_streaming.py',
    }
    import re
    if (
        'candidate/metadata.json' not in candidate_names
        or any(name != 'candidate/metadata.json' and not re.fullmatch(r'candidate/(train|val)_\d{6}\.npz', name) for name in candidate_names)
        or source_names != expected_source_names
        or set(names) != candidate_names | expected_source_names | {'train_candidate.py', 'manifest.json'}
    ):
        raise ValueError('Bundle contains files outside the Colab export allow-list')
    for name in names:
        if name.startswith('/') or '..' in Path(name).parts or '\\' in name:
            raise ValueError(f'Unsafe bundle member path: {name}')
    helper = zf.read('train_candidate.py')
    for name, details in manifest['files'].items():
        digest = hashlib.sha256()
        size = 0
        with zf.open(name) as member:
            for chunk in iter(lambda: member.read(1024 * 1024), b''):
                digest.update(chunk)
                size += len(chunk)
        if size != details['bytes'] or digest.hexdigest() != details['sha256']:
            raise ValueError(f'Bundle integrity check failed for {name}')

helper_path = Path('/content/train_candidate.py')
helper_path.write_bytes(helper)
sys.path.insert(0, '/content')
import train_candidate as training_helpers
benchmark_candidate = training_helpers.benchmark_candidate
extract_bundle = training_helpers.extract_bundle
train_candidate = training_helpers.train_candidate

if (BUNDLE_DIR / 'candidate' / 'metadata.json').is_file():
    print('Bundle is already extracted at:', BUNDLE_DIR)
    bundle_dir = BUNDLE_DIR
else:
    bundle_dir = extract_bundle(BUNDLE_ZIP, BUNDLE_DIR, exist_ok=True)

print('Verified bundle ready at:', bundle_dir)
print('Run/checkpoint/model outputs will be written to:', RUN_DIR)

In [ ]:
# Short benchmark only: this does not start full training or save a model.
benchmark = benchmark_candidate(bundle_dir, max_batches=16)

## Optional full training — explicit opt-in required

Review the measured runtime and confirmed storage before deciding to train.
- **Full training disabled by default:** Leave `RUN_FULL_TRAINING = False` to stop after benchmarking.
- **Artifact safety:** The candidate artifact is named `collaborative_ncf_candidate_20261011` and saved to `RUN_DIR/models/collaborative_ncf_candidate_20261011/`. It never touches production weights.
- **New run (`RESUME = False`):** Starts training from epoch 1. Checkpoints are saved atomically at each completed epoch boundary to `RUN_DIR/checkpoints/candidate_epoch.pt`.
- **Protection against silent resets:** If a checkpoint already exists at `RUN_DIR/checkpoints/candidate_epoch.pt` and `RESUME = False`, training will raise `FileExistsError` to prevent accidentally overwriting or silently restarting from epoch 1.
- **Resuming (`RESUME = True`):** Loads `candidate_epoch.pt`, verifies the dataset fingerprint, model, and optimizer configurations, and resumes from `start_epoch + 1`.
- **IMPORTANT — `/content` does NOT persist across runtime resets:**
  In Drive-free mode, local `/content` storage is erased if Colab disconnects or recycles the runtime.
  To preserve progress, use the download cell below after completed epochs to download `candidate_epoch.pt` to your local machine.
  To resume in a new session: re-upload `cinemind_ncf_candidate_20261011.zip` to `/content/`, create `/content/cinemind_run/checkpoints/`, upload your saved `candidate_epoch.pt` there, and run with `RESUME = True`.

In [ ]:
RUN_FULL_TRAINING = False
RESUME = False

if not RUN_FULL_TRAINING:
    print('Full training is disabled. Set RUN_FULL_TRAINING = True only after reviewing the benchmark.')
else:
    confirmation = input('Type TRAIN CANDIDATE to start/resume the GPU training run: ')
    if confirmation != 'TRAIN CANDIDATE':
        raise RuntimeError('Confirmation did not match; no training was started.')
    report = train_candidate(bundle_dir, RUN_DIR, resume=RESUME)

In [ ]:
report_path = RUN_DIR / 'training_report.json'
if report_path.is_file():
    import json
    print(json.dumps(json.loads(report_path.read_text()), indent=2))
else:
    print('No training report exists. Benchmarking alone does not produce evaluation results.')

In [ ]:
# Checkpoint and Model Export Helpers
# Note: /content does NOT persist after a Colab runtime disconnect or reset.
# Use download_checkpoint() periodically to save progress to your local machine.
from google.colab import files
import shutil

def download_checkpoint():
    """Download the latest epoch checkpoint to your local machine before session ends."""
    ckpt = RUN_DIR / 'checkpoints' / 'candidate_epoch.pt'
    if ckpt.is_file():
        print(f'Downloading checkpoint ({ckpt.stat().st_size / 1024**2:.1f} MB)...')
        files.download(str(ckpt))
    else:
        print(f'No checkpoint file found at: {ckpt}')

def download_candidate_results():
    """Package and download the trained candidate model and evaluation report."""
    if not (RUN_DIR / 'training_report.json').is_file():
        print('Training report not found. Complete training first before downloading results.')
        return
    out_archive = Path('/content/cinemind_ncf_candidate_results')
    archive_path = shutil.make_archive(str(out_archive), 'zip', RUN_DIR)
    print(f'Downloading results archive ({Path(archive_path).stat().st_size / 1024**2:.1f} MB)...')
    files.download(archive_path)

# Uncomment and run to download:
# download_checkpoint()
# download_candidate_results()